# Bivariate VAR PhiID Noise Sweep

This notebook reproduces a simple bivariate VAR(1) simulation, computes `STS` and `RTR` under both `MMI` and `CCS`, and plots how those atoms change as observation noise increases.

Important modelling choice: the sweep is over **observation noise** rather than only scaling the latent innovation variance. `PhiIDFull` internally standardizes each channel, so a pure innovation-amplitude rescaling is often a weak manipulation after normalization. Adding observation noise directly changes the effective signal-to-noise ratio of the lagged dependencies.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO = Path('/Users/borjan/CNRS/projects/TVBToolkit').resolve()
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

from tvbtoolkit.analysis import load_var_noise_sweep, summarize_sweep, sweep_long_form, plot_noise_sweep_publication

OUTPUT_ROOT = REPO / 'results' / 'phiid_var_bivariate_noise_sweep'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT

In [ ]:
cmd = [
    sys.executable,
    str(REPO / 'scripts' / 'run_bivariate_var_phiid_noise_sweep.py'),
    '--output-root', str(OUTPUT_ROOT),
    '--noise-levels', '0.0', '0.05', '0.1', '0.2', '0.4', '0.8', '1.2',
    '--n-replicates', '32',
    '--n-timepoints', '1200',
    '--burnin', '300',
    '--self-coef', '0.72',
    '--cross-coef', '0.18',
    '--run-matlab',
    '--matlab-parallel',
    '--matlab-workers', '8',
]
# subprocess.run(cmd, check=True)
cmd

In [ ]:
results = load_var_noise_sweep(OUTPUT_ROOT / 'phiid_var_noise_sweep.mat')
raw_df = sweep_long_form(results)
summary_df = summarize_sweep(raw_df)
summary_df.head()

In [ ]:
fig, axes = plot_noise_sweep_publication(summary_df)
fig

## Interpretation guide

- `STS` reflects synergistic predictive information in the bivariate time-delayed system.
- `RTR` reflects redundant predictive information.
- `MMI` usually behaves as the smoother, bottleneck-style overlap estimate.
- `CCS` is stricter and sign-sensitive, so it can suppress overlap when local informational contributions disagree.

A useful next extension is to sweep `common_noise_fraction` as well, to compare private observation noise with shared observation noise.